In [ ]:
# Process XML files to extract structured intervention data
import xml.etree.ElementTree as ET
from pathlib import Path
import json
from datetime import datetime
import logging
import os
import shutil

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger('xml_processor')

def extract_text_content(element):
    """Extract all text content from an element and its children."""
    text_parts = []
    if element.text and element.text.strip():
        text_parts.append(element.text.strip())
    for child in element:
        text_parts.append(extract_text_content(child))
        if child.tail and child.tail.strip():
            text_parts.append(child.tail.strip())
    return ' '.join(filter(None, text_parts))

def process_intervention(intervention):
    """Process a single intervention into a structured format."""
    orateur = intervention.find('.//ORATEUR')
    # Remove the language check to process all interventions regardless of language
    if not orateur:
        return None
        
    # Extract speaker info
    speaker_data = {
        'name': orateur.get('LIB'),
        'political_group': orateur.get('PP'),
        'language': orateur.get('LG'),
        'mep_id': orateur.get('MEPID'),
        'speaker_type': orateur.get('SPEAKER_TYPE')
    }
    
    # Extract paragraphs
    paragraphs = []
    for para in intervention.findall('.//PARA'):
        text = extract_text_content(para)
        if text.strip():
            paragraphs.append(text.strip())
            
    return {
        'speaker': speaker_data,
        'paragraphs': paragraphs
    }

# Get all XML files from input/xml directory
xml_dir = Path('/workspaces/kg/input/xml')

if not xml_dir.exists():
    logger.error(f"Directory {xml_dir} does not exist")
    raise FileNotFoundError(f"Directory {xml_dir} does not exist")

xml_files = list(xml_dir.glob('*.xml'))

if not xml_files:
    logger.error(f"No XML files found in {xml_dir}")
    raise FileNotFoundError(f"No XML files found in {xml_dir}")

logger.info(f"Found {len(xml_files)} XML files: {', '.join(x.name for x in xml_files)}")

# Create output directory for individual text files
output_dir = Path('input/txt/proceedings')
output_dir.mkdir(parents=True, exist_ok=True)

# Process each XML file and save as individual text files
total_interventions = 0
for xml_path in xml_files:
    try:
        # Parse XML file
        logger.info(f"Processing {xml_path.name}...")
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # Process interventions from this file
        file_interventions = []
        for intervention in root.findall('.//INTERVENTION'):
            processed = process_intervention(intervention)
            if processed:
                file_interventions.append(processed)
        
        # Skip if no interventions found
        if not file_interventions:
            logger.info(f"No interventions found in {xml_path.name}")
            continue
                
        logger.info(f"Extracted {len(file_interventions)} interventions from {xml_path.name}")
        total_interventions += len(file_interventions)
        
        # Create output file name based on the XML file name
        output_filename = xml_path.stem + ".txt"
        output_path = output_dir / output_filename
        
        # Write interventions to individual text file
        with open(output_path, 'w', encoding='utf-8') as f:
            # Write each intervention with speaker name and political group concatenated with speech
            for intervention in file_interventions:
                speaker_name = intervention['speaker']['name']
                political_group = intervention['speaker']['political_group']
                
                # Combine all paragraphs into a single text
                speech_text = ' '.join(intervention['paragraphs'])
                
                # Format the output as "Name said from the Political group that ... text"
                # Each intervention is now written as a single line without the extra newline
                if political_group:
                    f.write(f"{speaker_name} said from the {political_group} that {speech_text}\n")
                else:
                    f.write(f"{speaker_name} said from the NULL that {speech_text}\n")
        
        logger.info(f"Saved {len(file_interventions)} interventions to {output_path}")
        
        # Also copy the file to input/txt directory to ensure we have a copy there
        input_txt_dir = Path('input/txt/proceedings')
        input_txt_dir.mkdir(exist_ok=True)
        txt_copy_path = input_txt_dir / output_filename
        shutil.copy2(output_path, txt_copy_path)
        logger.info(f"Copied proceedings to {txt_copy_path}")
        
    except ET.ParseError as e:
        logger.error(f"Failed to parse {xml_path.name}: {str(e)}")
        continue
    except Exception as e:
        logger.error(f"Error processing {xml_path.name}: {str(e)}")
        continue

logger.info(f"Processed {total_interventions} interventions across {len(xml_files)} files")

Found 13 XML files: CRE-9-2022-05-03-ITM-005_EN.xml, CRE-9-2022-06-08-ITM-003_EN.xml, CRE-9-2022-06-22-ITM-014_EN.xml, CRE-9-2022-07-05-ITM-004_EN.xml, CRE-9-2022-09-13-ITM-005_EN.xml, CRE-9-2022-12-13-ITM-006_EN.xml, CRE-9-2023-03-14-ITM-005_EN.xml, CRE-9-2023-04-19-ITM-007_EN.xml, CRE-9-2023-05-09-ITM-007_EN.xml, CRE-9-2023-06-13-ITM-004_EN.xml, CRE-9-2023-11-22-ITM-006_EN.xml, CRE-9-2024-02-07-ITM-006_EN.xml, CRE-9-2024-03-13-ITM-006_EN.xml
Processing CRE-9-2022-05-03-ITM-005_EN.xml...


/tmp/ipykernel_25142/876365289.py:29: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  if not orateur:
Extracted 13 interventions from CRE-9-2022-05-03-ITM-005_EN.xml
Saved 13 interventions to input/txt/proceedings/CRE-9-2022-05-03-ITM-005_EN.txt
Copied proceedings to input/txt/CRE-9-2022-05-03-ITM-005_EN.txt
Processing CRE-9-2022-06-08-ITM-003_EN.xml...
Extracted 12 interventions from CRE-9-2022-06-08-ITM-003_EN.xml
Saved 12 interventions to input/txt/proceedings/CRE-9-2022-06-08-ITM-003_EN.txt
Copied proceedings to input/txt/CRE-9-2022-06-08-ITM-003_EN.txt
Processing CRE-9-2022-06-22-ITM-014_EN.xml...
Extracted 15 interventions from CRE-9-2022-06-22-ITM-014_EN.xml
Saved 15 interventions to input/txt/proceedings/CRE-9-2022-06-22-ITM-014_EN.txt
Copied proceedings to input/txt/CRE-9-2022-06-22-ITM-014_EN.txt
Processing CRE-9-2022-07-05-ITM-004_EN.xml...
Extracted 13 interventi

In [2]:
from langchain_core.documents import Document
from langchain_google_community import GoogleTranslateTransformer

In [9]:
sample_text = """
Dear team,

"""

In [10]:
# !pip install langchain-google-community[translate]

In [11]:

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/root/.config/gcloud/application_default_credentials.json"

documents = [Document(page_content=sample_text)]
translator = GoogleTranslateTransformer(project_id="bible-chat-58a44")

In [12]:
translated_documents = translator.transform_documents(
    documents, target_language_code="es"
)

print(translated_documents)

[Document(metadata={'model': '', 'detected_language_code': 'en'}, page_content='\nEstimado equipo,\n\n')]
